# ML Pipeline Demo

This notebook demonstrates the **full demo flow** of the project:

1. Download the last 6 months of hourly market data from Binance
2. Preprocess ETH, BTC, and ETHBTC into a single feature dataframe
3. Load the training and test configs
4. Prepare a notebook-friendly runtime that disables model persistence
5. Run the grid-search pipeline
6. Inspect selected models and example predictions

> **Note:** the demo version disables model saving, log files, and Excel outputs.  

## 0. Optional setup for Google Colab

If you run this notebook **inside the repository**, you can skip the next cell.

If you open the notebook directly in **Colab from GitHub**, set `REPO_URL` and run the setup cell once so the rest of the project files become available in the runtime.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

IS_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/ixor911/ML_Pipeline_Demo.git"

if IS_COLAB and not Path("core").exists():
    if not REPO_URL:
        raise ValueError("Set REPO_URL before running this cell in Colab.")
    subprocess.run(["git", "clone", REPO_URL, "repo"], check=True)
    os.chdir("repo")
    print("Repository cloned into Colab runtime.")
else:
    print("Repository files already available. Skipping clone step.")

## 1. Download raw market data

We fetch the last 6 months of **ETHUSDT**, **BTCUSDT**, and **ETHBTC** hourly candles directly from Binance.

In [ ]:
import pandas as pd
from IPython.display import display

from core.features.DataProvider import DataProvider

end = pd.Timestamp.utcnow().floor("h")
start = end - pd.DateOffset(months=6)

print(f"Downloading hourly candles from {start} to {end}...")

eth_df = DataProvider.download_klines_months(symbol="ETHUSDT", months=6)
btc_df = DataProvider.download_klines_months(symbol="BTCUSDT", months=6)
ethbtc_df = DataProvider.download_klines_months(symbol="ETHBTC", months=6)

print(f"ETH rows    : {len(eth_df)}")
print(f"BTC rows    : {len(btc_df)}")
print(f"ETHBTC rows : {len(ethbtc_df)}")

display(eth_df.head())

## 2. Preprocess the raw candles

The preprocessing step combines ETH, BTC, and ETHBTC into one processed dataset with:
- ETH technical and candle features
- BTC context features
- ETHBTC relative-strength features
- cross-market features

In [ ]:
from core.features.preprocessor.Preprocessor import Preprocessor

processed_df = Preprocessor.preprocess_and_save(
    eth_df=eth_df,
    btc_df=btc_df,
    ethbtc_df=ethbtc_df,
    months=6
)

print(f"Processed rows    : {len(processed_df)}")
print(f"Processed columns : {len(processed_df.columns)}")
display(processed_df.head())

## 3. Load training and test configs

Here we load:

- `model_train_basic_grid` — the training grid
- `model_test_basic` — the test slicing config

In [ ]:
from pprint import pprint

from core.io.loader import load_config, load_config_grid

training_configs = list(load_config_grid("model_train_basic_grid"))
test_config = load_config("model_test_basic")

print(f"Training configs loaded: {len(training_configs)}")
print("\nExample training config:")
pprint(training_configs[0])

print("\nTest config:")
pprint(test_config)

## 4. Build test candles from the processed dataframe

The grid-search pipeline evaluates all candidates on one shared test candle window.
We build that window from the freshly processed dataset using the test slicing config.


In [ ]:
from core.target.slicing import slice_data

test_windows = slice_data(
    df=processed_df,
    slicing_config=test_config["slicing"],
)
test_candles = test_windows[0]

print(f"Test candles rows: {len(test_candles)}")
display(test_candles.head())


## 5. Run the grid-search pipeline

This step:
- trains models for all training configs combinations
- expands each trained model into category-specific candidates
- evaluates them on the shared test candles
- keeps only the strongest candidates in `models_ram`


In [ ]:
import os

from core.pipelines.grid_search_pipeline import (
    grid_search_pipeline,
    print_models_ram_metrics,
)

LIMIT = 3 # limit models per category

models_ram = {}

models_ram = grid_search_pipeline(
    training_configs=training_configs,
    test_candles=test_candles,
    models_ram=models_ram,
    limit=LIMIT
)

print("Grid search finished.")


## 6. Inspect the final in-memory model pool

In [ ]:
def count_models(models_ram: dict) -> int:
    return sum(len(category_models) for category_models in models_ram.values())

def count_categories(models_ram: dict) -> int:
    return len(models_ram)

print(f"Categories kept : {count_categories(models_ram)}")
print(f"Models kept     : {count_models(models_ram)}")

print_models_ram_metrics(models_ram=models_ram)


## 7. What this demo notebook shows

This notebook demonstrates the main ML workflow of the demo project:

- downloading recent market data
- preprocessing ETH / BTC / ETHBTC candles
- loading train/test configs
- running grid search
- keeping the strongest candidate models in memory
- inspecting final predictions

The full private version of the project also includes:
- model persistence and loading
- model metadata analysis
- backtesting and grid-backtesting
- system-management helper pipelines
- ongoing work on step-by-step retraining and AI-agent integration
